In [2]:
import subprocess
import sys

# Pins mirrored from PINS.md
VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [3]:
!python --version
!nvidia-smi --query-gpu=name --format=csv,noheader

Python 3.12.13
Tesla T4


In [4]:
pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
)

print("serving pins installed")

installing: vllm==0.6.* transformers==4.46.* accelerate==1.1.* httpx==0.27.* openai==1.54.*
serving pins installed


In [5]:
import os
import signal
import subprocess

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
}

def build_cmd(args: dict) -> list:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]

    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]

    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)

    print("launching:", " ".join(cmd))

    logf = open(SERVER_LOG, "wb")

    proc = subprocess.Popen(
        cmd,
        stdout=logf,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000
server pid 2394, logging to /content/server.log


In [6]:
import json
from google.colab import files

# Upload baselines.json
uploaded = files.upload()

baseline = json.load(open("baselines.json"))
print("baseline batch tokens/s:", baseline["batch"])

Saving baselines.json to baselines.json
baseline batch tokens/s: {'1': 33.0, '4': 46.7, '8': 92.7}


In [7]:
# Async A/B client for Lab W3D3 (engine swap).
# Paste the whole file as one Colab cell after the vLLM server is healthy.
# It fires concurrent chat completions at each level, excludes warm-up requests,
# and reports aggregate tokens/s.

import asyncio
import time

import httpx


# Fixed prompts so every run measures the same work.
FIXED_PROMPTS = [
    "In one sentence, what is a GPU?",
    "List three reasons decode is memory-bound.",
    "Explain the KV cache to a new ops engineer in two sentences.",
    "What does continuous batching change versus static batching?",
    "Give a one-line definition of tokens per second.",
    "Why does a longer prompt increase time to first token?",
    "Name two things quantisation trades away for smaller memory.",
    "Summarise what an inference server does in three short bullets.",
]


# Output lengths for the requests.
# This matches Monday's mixed-length workload.
QUEUE = [32, 32, 32, 256] * 6

# Default output length if one is not supplied.
MAX_TOKENS = 128

# Warm-up requests are excluded from the measurements.
WARMUP = 4


async def _one_request(
    client,
    base_url,
    model,
    prompt,
    max_tokens=MAX_TOKENS
):
    """Send one chat completion and return completion token count."""

    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": False,
    }

    r = await client.post(
        f"{base_url}/chat/completions",
        json=payload
    )

    r.raise_for_status()

    body = r.json()
    usage = body.get("usage", {})

    # Prefer the server's reported completion token count.
    ct = usage.get("completion_tokens")

    # Fallback if usage is unavailable.
    if ct is None:
        ct = len(
            body["choices"][0]["message"]["content"].split()
        )

    return ct


async def _run_level(
    client,
    base_url,
    model,
    prompts,
    concurrency,
    total_requests
):
    """Run requests with at most `concurrency` active at once."""

    sem = asyncio.Semaphore(concurrency)
    counts = []

    async def guarded(prompt, max_tokens):
        async with sem:
            return await _one_request(
                client,
                base_url,
                model,
                prompt,
                max_tokens
            )

    # Create the requests using the fixed prompt/output queue.
    tasks = [
        asyncio.create_task(
            guarded(
                prompts[i % len(prompts)],
                QUEUE[i % len(QUEUE)]
            )
        )
        for i in range(total_requests)
    ]

    # Start timing only after all tasks have been created.
    t0 = time.time()

    for coro in asyncio.as_completed(tasks):
        counts.append(await coro)

    dt = time.time() - t0
    total_tokens = sum(counts)

    return {
        "concurrency": concurrency,
        "requests": total_requests,
        "tokens_per_s": round(total_tokens / dt, 1),
        "wall_s": round(dt, 3),
    }


async def run_sweep(
    base_url,
    model,
    prompts=FIXED_PROMPTS,
    concurrencies=(1, 4, 8),
    requests_per_level=24
):
    """Run the concurrency sweep and return the measured results."""

    results = []

    async with httpx.AsyncClient(timeout=120.0) as client:

        # Warm-up requests.
        # These are intentionally excluded from the measurements.
        await asyncio.gather(*[
            _one_request(
                client,
                base_url,
                model,
                prompts[i % len(prompts)]
            )
            for i in range(WARMUP)
        ])

        # Run each concurrency level.
        for c in concurrencies:

            level = await _run_level(
                client,
                base_url,
                model,
                prompts,
                c,
                requests_per_level
            )

            print("level:", level)
            results.append(level)

    return results

In [8]:
import urllib.request
import urllib.error

try:
    with urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5) as r:
        print("Server is up:", r.status)
        print(r.read().decode()[:500])
except Exception as e:
    print("Server is NOT up")
    print(type(e).__name__, e)

Server is NOT up
URLError <urlopen error [Errno 111] Connection refused>


In [11]:
print("server object:", server if "server" in globals() else "NOT DEFINED")

server object: <Popen: returncode: None args: ['/usr/bin/python3', '-m', 'vllm.entrypoints....>


In [12]:
!ps aux | grep -i vllm


root        2394 41.0 10.4 6240672 1382360 ?     Ssl  10:43   0:16 /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000
root        2567 88.0  2.8 3373724 381976 ?      Rl   10:43   0:02 /usr/bin/python3 -m vllm.model_executor.models.registry
root        2584  0.0  0.0   7376  3540 ?        S    10:43   0:00 /bin/bash -c ps aux | grep -i vllm
root        2586  0.0  0.0   6484  2400 ?        S    10:43   0:00 grep -i vllm


In [13]:
import os

print("Log exists:", os.path.exists("/content/server.log"))

if os.path.exists("/content/server.log"):
    with open("/content/server.log", "r", errors="replace") as f:
        lines = f.readlines()

    print("".join(lines[-50:]))
else:
    print("No server.log found.")

Log exists: True
2026-09-02 10:43:35.207153: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 09-02 10:43:51 api_server.py:712] vLLM API server version 0.6.6.post1
INFO 09-02 10:43:51 api_server.py:713] args: Namespace(host=None, port=8000, uvicorn_log_level='info', allow_credentials=False, allowed_origins=['*'], allowed_methods=['*'], allowed_headers=['*'], api_key=None, lora_modules=None, prompt_adapters=None, chat_template=None, chat_template_content_format='auto', response_role='assistant', ssl_keyfile=None, ssl_certfile=None, ssl_ca_certs=None, ssl_cert_reqs=0, root_path=None, middleware=[], return_tokens_as_token_ids=False, disable_frontend_multiprocessing=False, enable_request_id_headers=False, enable_auto_tool_choice=False, tool_call_

In [14]:
import time
import urllib.request
import urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s

    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(
                        f"server healthy after about {waited}s: "
                        f"{url} -> 200"
                    )
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass

        time.sleep(interval_s)

    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())

    return False

healthy = wait_for_health()

server healthy after about 143s: http://localhost:8000/v1/models -> 200


In [15]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

r = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "In one sentence, what is a GPU?"
        }
    ],
)

print(r.choices[0].message.content)

A GPU, or Graphics Processing Unit, is a specialized processor designed to accelerate computations involved in rendering graphics and video content on electronic devices.


In [16]:
prompts = FIXED_PROMPTS

vllm_measured = await run_sweep(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompts=prompts,
    concurrencies=[1, 4, 8],
)

for level in vllm_measured:
    print(level)

level: {'concurrency': 1, 'requests': 24, 'tokens_per_s': 40.9, 'wall_s': 33.924}
level: {'concurrency': 4, 'requests': 24, 'tokens_per_s': 103.9, 'wall_s': 13.374}
level: {'concurrency': 8, 'requests': 24, 'tokens_per_s': 161.1, 'wall_s': 8.623}
{'concurrency': 1, 'requests': 24, 'tokens_per_s': 40.9, 'wall_s': 33.924}
{'concurrency': 4, 'requests': 24, 'tokens_per_s': 103.9, 'wall_s': 13.374}
{'concurrency': 8, 'requests': 24, 'tokens_per_s': 161.1, 'wall_s': 8.623}


In [17]:
import json

def tokps_at(level_list, c):
    return next(
        x["tokens_per_s"]
        for x in level_list
        if x["concurrency"] == c
    )

vllm_by_c = {
    x["concurrency"]: x["tokens_per_s"]
    for x in vllm_measured
}

base_by_c = {
    int(k): v
    for k, v in baseline["batch"].items()
}

speedup = {
    c: round(
        vllm_by_c[c] / base_by_c[c],
        2
    )
    for c in vllm_by_c
    if c in base_by_c
}

report = {
    "baseline": base_by_c,
    "vllm": vllm_by_c,
    "speedup_by_concurrency": speedup,
    "predicted_speedup": 1.5,
}

with open("ab_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

{
  "baseline": {
    "1": 33.0,
    "4": 46.7,
    "8": 92.7
  },
  "vllm": {
    "1": 40.9,
    "4": 103.9,
    "8": 161.1
  },
  "speedup_by_concurrency": {
    "1": 1.24,
    "4": 2.22,
    "8": 1.74
  },
  "predicted_speedup": 1.5
}


In [18]:
static_scaling = base_by_c[8] / base_by_c[1]
vllm_scaling = vllm_by_c[8] / vllm_by_c[1]

print(
    f"static batching scales {static_scaling:.2f}x, "
    f"vLLM scales {vllm_scaling:.2f}x"
)

print(
    f"continuous batching is worth "
    f"{vllm_scaling / static_scaling:.2f}x of scaling"
)

static batching scales 2.81x, vLLM scales 3.94x
continuous batching is worth 1.40x of scaling


In [19]:
print("=== W3D3 HEADLINE ===")
print(f"Concurrency-8 vLLM throughput: {vllm_by_c[8]} tokens/s")
print(f"Monday batch-8 throughput: {base_by_c[8]} tokens/s")
print(f"Speedup at concurrency 8: {speedup[8]}x")
print(f"Static scaling: {static_scaling:.2f}x")
print(f"vLLM scaling: {vllm_scaling:.2f}x")

=== W3D3 HEADLINE ===
Concurrency-8 vLLM throughput: 161.1 tokens/s
Monday batch-8 throughput: 92.7 tokens/s
Speedup at concurrency 8: 1.74x
Static scaling: 2.81x
vLLM scaling: 3.94x


In [22]:
import os
import signal
import time
import urllib.request
import urllib.error

def shutdown_server(proc=None, port=PORT):
    try:
        proc = server if proc is None else proc

        os.killpg(
            os.getpgid(proc.pid),
            signal.SIGTERM
        )

        print(
            f"sent SIGTERM to process group of pid {proc.pid}"
        )

    except (ProcessLookupError, NameError):
        print("no server process to kill")

    time.sleep(3)

    try:
        with urllib.request.urlopen(
            f"http://localhost:{port}/v1/models",
            timeout=2
        ):
            print(
                f"WARNING: port {port} still answering; "
                f"something is still up"
            )

    except (
        urllib.error.URLError,
        ConnectionError,
        OSError
    ):
        print(f"port {port} is free")

shutdown_server()

sent SIGTERM to process group of pid 2394
port 8000 is free


In [23]:
# Green-check verifier for Lab W3D3 (engine swap).
# Paste this as the last cell of your day-3 notebook and run it.

import json
import os


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    path = "ab_report.json"

    if not os.path.exists(path):
        fail(f"{path} not found; write it in Cell 5")

    try:
        with open(path) as fh:
            report = json.load(fh)

    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")

    for key in ("baseline", "vllm", "speedup_by_concurrency"):
        if key not in report:
            fail(f"{path} missing key: {key}")

    baseline = report["baseline"]
    vllm = report["vllm"]
    speedup = report["speedup_by_concurrency"]

    if not isinstance(baseline, dict) or not baseline:
        fail(
            "baseline must be a non-empty object "
            "(from Monday's baselines.json)"
        )

    if not isinstance(vllm, dict) or not vllm:
        fail(
            "vllm must be a non-empty object "
            "of measured throughput"
        )

    if not isinstance(speedup, dict) or not speedup:
        fail(
            "speedup_by_concurrency must be a non-empty object"
        )

    def get_c(d, c):
        for k, v in d.items():
            if str(k) == str(c):
                return v
        return None

    base8 = get_c(baseline, 8)
    vllm8 = get_c(vllm, 8)

    if base8 is None:
        fail(
            "baseline has no concurrency-8 "
            "(batch-8) number"
        )

    if vllm8 is None:
        fail(
            "vllm has no concurrency-8 number"
        )

    if not isinstance(base8, (int, float)) or not isinstance(vllm8, (int, float)):
        fail(
            "concurrency-8 throughput values must be numbers"
        )

    if not vllm8 > base8:
        fail(
            f"vllm concurrency-8 throughput ({vllm8}) "
            f"not above baseline batch-8 ({base8}); "
            f"the engine swap should win here"
        )

    s8 = get_c(speedup, 8)

    if s8 is None or not isinstance(s8, (int, float)):
        fail(
            "speedup_by_concurrency has no numeric "
            "value at concurrency 8"
        )

    expected = vllm8 / base8

    if abs(s8 - expected) > 0.1:
        fail(
            f"speedup at 8 ({s8}) does not match "
            f"vllm/baseline ({expected:.2f}); "
            f"recompute it"
        )

    print(
        f"baseline batch-8: {base8}, "
        f"vllm concurrency-8: {vllm8}"
    )

    print(f"speedup at 8: {s8}x")
    print("GREEN CHECK: PASS")


try:
    main()

except _Stop:
    try:
        get_ipython()
    except NameError:
        raise SystemExit(1)

baseline batch-8: 92.7, vllm concurrency-8: 161.1
speedup at 8: 1.74x
GREEN CHECK: PASS
